### **Import Data** and preview Data shapes

### Initialize the Environment:
##### Virtual Environment Commands

| Command | Linux/Mac | GitBash |
| ------- | --------- | ------- |
| Create | `python3 -m venv venv` | `python -m venv venv` |
| Activate | `source venv/bin/activate` | `source venv/Scripts/activate` |
| Install | `pip install -r requirements.txt` | `pip install -r requirements.txt` |
| Deactivate | `deactivate` | `deactivate` |
##### Select the Kernel (This will be in the Requirements.txt eventually)

Using the venv (Python 3.13.2) located  in venv/bin/python)

### **Project Overview & Plan**
Capstone Project for Code:You Data Analysis track. This project analyzes Beer Recipes for frequency of uploads for various beer styles, while capturing preferences of strength, hopiness and batch size.    The goal of the project is to demonstrate a general knowledge of Python (Pandas, Numpy, MatLibPlot, Plotly), SQL(MySQL), Tableu, Cursor and ChatGPT.
**Data Sources:**

The datasets used in this project are all related to online beer recipes. One dataset contains the different styles of beer as recognized by the Beer Judge Certification Program (BJCP.org).
- [Beersmith Recipes](https://beersmithrecipes.com/recent/) - scraped data that contains certain fields of 100% of the all grain beer recipes that have been uploaded by users.

- [Brewers Friend All-Grain Recipes](https://www.brewersfriend.com/homebrew-recipes/all-grain/) - scraped data that contains select fields of 100% of the all-grain beer recipes that have been uploaded by users.
- [Kaggle - Brewers Friend Recipes](https://www.kaggle.com/datasets/jtrofe/beer-recipes) - data from Kaggle that contains a subset of beer recipes.
- [BJCP - Judging Styles of Beer](https://github.com/ascholer/bjcp-styleview/blob/main/styles.json) - dataset that contains the criterea used to judge beer. Will help determine if recipes meet the criterea to be considered a specific style of beer.

### **Import Data** and preview Data shapes

In [16]:
import pandas as pd
# import numpy as np
import re # Imported "re" Python module to assist with parsing the data with pattern matching
# import matplotlib
from pandas import DataFrame
from typing import Optional # received warning about "float | None" when my type hint only included "float"
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FuncFormatter
# from rich.console import Console
# from rich.table import Table

In [17]:
bs_df = pd.read_csv('beersmith_recipes.csv')
bf_df = pd.read_csv('bf_recipes.csv')
kag_df = pd.read_csv('recipeData.csv', encoding='ISO-8859-1')
styles_df = pd.read_json('styles.json')
print(bf_df.shape)
print(bs_df.shape)
print(kag_df.shape)
print(styles_df.shape)

(215580, 8)
(63121, 5)
(73861, 23)
(116, 28)


### Data Cleanup

#### **Beer Smith Recipes -** beersmith_recipes.csv = bs_df
Will analyze data, delete duplicates, remove what appears to be test data, resolve missing data, split the stats column into the individual components to match up with the Brewers friend data. While cleaning up the data, will create a function to clean up the data in the other datasets. In the Beer Smith data there were 31 rows missing the Recipe Name. Since the name wasn't crucial to the analysis of the recipes, but the recipe information was still of value, we assigned a generic name to each of the rows missing the Recipe Name.

**Determine Null Values**

In [18]:
# Create a copy for cleaning the dataframe
bs_df_cleaned = bs_df.copy()

# Count null values for each column
null_counts = bs_df_cleaned.isnull().sum()

# Filter columns with null counts greater than zero
columns_with_nulls = null_counts[null_counts > 0].index.tolist()

print(null_counts)
print(columns_with_nulls)


Recipe Name    31
Recipe URL      0
Beer Style      0
Brewer          0
Stats           0
dtype: int64
['Recipe Name']


**Replace Null Values**

In [19]:
# This function is to replace null values in the Recipe Name Column with a Generic
# unique name that begins with the first 2 characters of the dataframe name and is 
# incremented by 1 to keep the name unique. This was done because the data in the 
# rest of the columns contributed to the data and analysis.
# This code was written with a lot of back and forth with Perplexity.ai.

import inspect

def replace_column_nulls(df, target_column):
    # Replace nulls in one specific column with unique numbered values
    # Get dataframe variable name safely
    try:
        caller_frame = inspect.currentframe().f_back
        df_name = [k for k,v in caller_frame.f_locals.items() if v is df][0]
        prefix = df_name[:2].upper()
    except:
        prefix = "DF"

    # Create column-specific generator
    def col_generator():
        counter = 1
        while True:
            yield f"{prefix} No Name {counter}"
            counter += 1

    # Only process specified column
    mask = df[target_column].isnull()
    num_nulls = mask.sum()
    
    if num_nulls > 0:
        gen = col_generator()
        replacements = [next(gen) for _ in range(num_nulls)]
        df.loc[mask, target_column] = replacements
        
        # Show changes
        print(f"Replaced {num_nulls} nulls in {target_column}")
        print("Example replacement:", replacements[0])
    
    return df

In [20]:
# Call the function
bf_df_cleaned = replace_column_nulls(bs_df_cleaned, 'Recipe Name')

Replaced 31 nulls in Recipe Name
Example replacement: BS No Name 1


**Delete Duplicates and rows with no usable data in the Beer Style column**

In [21]:
def clean_bs_df(df) -> DataFrame :
    """
    Further Clean the Beer Smith DataFrame by deleting duplicates and eliminating 
    rows with no value in the 'Beer Style' column.

    Parameters:
   (pd.DataFrame): The dataframe to be cleaned

    Returns:
    pd.DataFrame: The cleaned Beer Smith DataFrame.
    """
    # Create a copy for cleaning the dataframe
    # bs_df_cleaned = bs_df.copy()

    # Delete all duplicates based on the value in the column "Recipe URL"
    df.drop_duplicates(subset=['Recipe URL'], keep='first', inplace=True, ignore_index=True)

    # Eliminate rows with no value in column "Beer Style". In Data Wrangler I noticed columns with only a pair of parentheses in the column. 
    df = df[df['Beer Style'] != '()']

    return df


In [22]:
# Clean the Beer Smith DataFrame
bs_df_cleaned = clean_bs_df(bs_df_cleaned)
print(bs_df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
Index: 56761 entries, 0 to 56767
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Recipe Name  56761 non-null  object
 1   Recipe URL   56761 non-null  object
 2   Beer Style   56761 non-null  object
 3   Brewer       56761 non-null  object
 4   Stats        56761 non-null  object
dtypes: object(5)
memory usage: 2.6+ MB
None


#### Splitting the Stats & Beer Style columns into individual columns of Data: 
Beer Style "Dark Mild (13A)"
Stats "OG: 1.073 (17.7° P), Bitterness: 34.5 IBUs, ABV: 6.9 %"

| Style|Style Number|
|:----------:|:----------:|
| Dark Mild| 13A| 

| OG| Plato| IBU| ABV|FG |
|:----------:|:----------:|:----------:|:----------:|:----------:|
| 1.073| 17.7| 34.5 | 6.9 |Calc|


In [23]:
def parse_columns(df: pd.DataFrame) -> pd.DataFrame:
    def extract_values(row: pd.Series) -> pd.Series:
        stats = row['Stats']
        beer_style = row['Beer Style']
        
        style_match = re.match(r'(.*?)\s*\((.*?)\)', beer_style)
        
        return pd.Series({
            'Style': style_match.group(1).strip() if style_match else None,
            'Style Number': style_match.group(2) if style_match else None,
            'OG': extract_float(stats, r'OG: (\d+\.\d+)'),
            'Plato': extract_float(stats, r'(\d+\.\d+)° P'),
            'IBU': extract_float(stats, r'Bitterness: (\d+\.\d+)'),
            'ABV': extract_float(stats, r'ABV: (\d+\.\d+)'),
        })

# was getting a warning about "float | None" and Perplexity.ai explained why and how to fix

    def extract_float(text: str, pattern: str) -> Optional[float]:
        match = re.search(pattern, text)
        return float(match.group(1)) if match else None

    new_columns = df.apply(extract_values, axis=1)
    df = pd.concat([df, new_columns], axis=1) # Add new extracted columns
    df = df.drop(columns=["Beer Style", "Stats", "Recipe URL", "Brewer"])  # Drop original columns
    

    return df

In [24]:
bs_df_cleaned = parse_columns(bs_df_cleaned)

#### Balling Formula for Final Gravity (FG) Calculation (ChatGPT)
To estimate the **Final Gravity (FG)** using the **Original Gravity (OG) in Plato (°P)** and the **Alcohol by Volume (ABV)**, we will use the **Balling formula**:


-Step 1: Calculate Residual Extract (re) in degrees Plato
re = (plato / 1.25) - (abv / 0.79)

-Step 2: Convert re from degrees Plato to Specific Gravity (SG)
fg = 1 + (re / (258.6 - ((re / 258.2) * 227.1)))


In [25]:
def calculate_final_gravity(plato, abv):
    """
    Calculate Final Gravity (FG) using the Balling formula.

    Parameters:
    plato (float): Original Gravity in degrees Plato.
    abv (float): Alcohol by volume percentage.
    re (float): Residual Extract in degrees Plato.

    Returns:
    float: Estimated Final Gravity (FG).
    """
    # Step 1: Calculate Residual Extract (RE) in degrees Plato
    re = (plato / 1.25) - (abv / 0.79)

    # Step 2: Convert RE from degrees Plato to Specific Gravity (SG)
    fg = 1 + (re / (258.6 - ((re / 258.2) * 227.1)))

    return fg


def add_final_gravity_column(df):
    """
    Add a new column 'FG' to the DataFrame with calculated Final Gravity.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'Plato' and 'ABV' columns.

    Returns:
    None: Modifies the DataFrame in place by adding a new column 'FG'.
    """
    # Apply the calculate_final_gravity function on every row, creating a new column
    # and locating it after the Plato column to match the Brewers Friend data frame
    plato_index = df.columns.get_loc('Plato')
    df.insert(
        loc=plato_index + 1,
        column='FG',
        value=df.apply(lambda row: round(calculate_final_gravity(row['Plato'], row['ABV']), 3), axis=1)
)

In [26]:
add_final_gravity_column(bs_df_cleaned)
print(bs_df_cleaned)

                                            Recipe Name               Style  \
0            Super Magnifico Mexican Lager 8g - Solo v1           Cream Ale   
1                          Clemens Honey Stout - 12 gal      Imperial Stout   
2                                       Modelo Especial        Vienna Lager   
3                                              GammaRay     New England IPA   
4      Rockaway Chocolate Peanut Butter Stout 2 Batch 2         Sweet Stout   
...                                                 ...                 ...   
56763                                     Garbage Brown  American Brown Ale   
56764                                        Mild (110)                Mild   
56765                                  Simcoe Mild (99)                Mild   
56766                 Malted Bliss, Wedding Barley Wine  English Barleywine   
56767                           More LPANE Goodness APA   American Pale Ale   

      Style Number     OG  Plato     FG   IBU   ABV

### Data Cleanup

#### **Brewers Friend Recipes -** bf_recipes.csv = bf_df
Will analyze data, delete duplicates, remove what appears to be test data, resolve missing data and rename columns to match up with the BeerSmith column names. In the Brewers friend data there are 14 null values in the title column. Since the name isn't crucial to the analysis of the recipes, but the recipe information is still of value, I  assigned a generic name to each of the rows missing the Title.

In [27]:
def clean_recipe_data(df, columns_to_drop, column_mapping, location, column_name, formula, location_2, column_name_2, new_column_2 ) -> pd.DataFrame:
    """
    Part of the cleanup:
        Creating a copy of dataframe with cleaned up data
        Removing unneeded columns
        Renaming columns to match with other dataframes
        Adding column with complex calculation
        

    Parameters:
    df: The Dataframes we are dropping columns
    columns_to_drop: the columns we don't need
    columns_to_rename: the columns to rename so they are common between the 2 data sets


    """
    # List of columns to drop by index
    # cols_to_drop = [2, 7, ]

    # Drop the specified columns
    df.drop(df.columns[columns_to_drop], axis=1, inplace=True)
    
    # Adding a new column named with a formula to create calculated value
    # match up with the other dataframes
    df.rename(columns=column_mapping, inplace=True)
    
    # Calculate the new column
    new_column = df.eval(formula)
    
    # Round up to 1 decimal place
    new_column = new_column.apply(lambda x: round(x, 1))
    
    # Insert the new column at the specified location with calculated value
    df.insert(location, column_name, new_column)
    
    # Insert another new column with no data for future calculations
    df.insert(location_2, column_name_2, new_column_2)
    # print(df.info)

    return df

In [28]:
# Columns to drop in bf_df_cleaned
cols_to_drop = [2, 7 ]

# Columns to rename and the new name
column_mapping = {
    'Title': 'Recipe Name'
}

# Add column for plato and calculate values to populate
location = 3
column_name = 'Plato'
formula = '-463.37 + (668.72 * OG) - (205.35 * OG**2)'

# Add column to match other recipe dataset
location_2 = 2
column_name_2 = 'Style Number'
new_column_2 = None


bf_df_cleaned = clean_recipe_data(
    bf_df, cols_to_drop, column_mapping, location, column_name,
    formula, location_2, column_name_2, new_column_2
)
bf_df_cleaned


,Recipe Name,Style,Style Number,OG,Plato,FG,ABV,IBU
0,Avg. Perfect Northeast IPA (NEIPA),Specialty IPA: New England IPA,None,1.062,15.2,1.013,6.50,59.26
1,Sierra Nevada Pale Ale Clone,American Pale Ale,None,1.055,13.6,1.013,5.58,39.79
2,Vanilla Cream Ale,Cream Ale,None,1.055,13.6,1.013,5.48,19.44
3,Zombie Dust Clone - ALL GRAIN,American IPA,None,1.061,15.0,1.016,5.94,62.42
4,Russian River Pliny The Elder (original),Imperial IPA,None,1.072,17.5,1.018,7.09,232.89
...,...,...,...,...,...,...,...,...
215575,Awesome Recipe,American IPA,None,1.054,13.3,1.010,5.82,19.23
215576,Dildo NEIPA,Specialty IPA: New England IPA,None,1.061,15.0,1.012,6.43,18.70
215577,Double Down,Specialty IPA: New England IPA,None,1.082,19.8,1.015,8.83,28.76
215578,AGAIN,American IPA,None,1.029,7.3,1.010,2.51,60.87


In [29]:
# Function to delete rows that don't contain information or have incorrect values
# Incorrect values are values that aren't possible. For example can't have an OG > 1.250
def delete_useless_rows(df, column_name, condition):
    # This function will delete rows that contain unusable data
    if isinstance(condition, str):
        # For string conditions, use str.contains
        df = df[~df[column_name].str.contains(condition, case=False, na=False)]
    else :
        df = df[~df[column_name].apply(condition)]
    return df

In [30]:
# Parameters for removing useless rows

# Removing when there is no style to work with
column_name = 'Style'
condition = 'No Profile Selected'
bf_df_cleaned = delete_useless_rows(bf_df_cleaned, column_name, condition)

# Removing rows when the data is out of range
column_name = 'OG'
condition = lambda x: x > 1.25
bf_df_cleaned = delete_useless_rows(bf_df_cleaned, column_name, condition)

bf_df_cleaned

,Recipe Name,Style,Style Number,OG,Plato,FG,ABV,IBU
0,Avg. Perfect Northeast IPA (NEIPA),Specialty IPA: New England IPA,None,1.062,15.2,1.013,6.50,59.26
1,Sierra Nevada Pale Ale Clone,American Pale Ale,None,1.055,13.6,1.013,5.58,39.79
2,Vanilla Cream Ale,Cream Ale,None,1.055,13.6,1.013,5.48,19.44
3,Zombie Dust Clone - ALL GRAIN,American IPA,None,1.061,15.0,1.016,5.94,62.42
4,Russian River Pliny The Elder (original),Imperial IPA,None,1.072,17.5,1.018,7.09,232.89
...,...,...,...,...,...,...,...,...
215575,Awesome Recipe,American IPA,None,1.054,13.3,1.010,5.82,19.23
215576,Dildo NEIPA,Specialty IPA: New England IPA,None,1.061,15.0,1.012,6.43,18.70
215577,Double Down,Specialty IPA: New England IPA,None,1.082,19.8,1.015,8.83,28.76
215578,AGAIN,American IPA,None,1.029,7.3,1.010,2.51,60.87


In [31]:
# This function is to replace null values in the Recipe Name Column with a Generic
# unique name that begins with the first 2 characters of the dataframe name and is 
# incremented by 1 to keep the name unique. This was done because the data in the 
# rest of the columns contributed to the data and analysis.
# This code was written with a lot of back and forth with Perplexity.ai.

import inspect

def replace_column_nulls(df, target_column):
    """Replace nulls in one specific column with unique numbered values"""
    # Get dataframe variable name safely
    try:
        caller_frame = inspect.currentframe().f_back
        df_name = [k for k,v in caller_frame.f_locals.items() if v is df][0]
        prefix = df_name[:2].upper()
    except:
        prefix = "DF"

    # Create column-specific generator
    def col_generator():
        counter = 1
        while True:
            yield f"{prefix} No Name {counter}"
            counter += 1

    # Only process specified column
    mask = df[target_column].isnull()
    num_nulls = mask.sum()
    
    if num_nulls > 0:
        gen = col_generator()
        replacements = [next(gen) for _ in range(num_nulls)]
        df.loc[mask, target_column] = replacements
        
        # Show changes
        print(f"Replaced {num_nulls} nulls in {target_column}")
        print("Example replacement:", replacements[0])
    
    return df

In [32]:
bf_df_cleaned = replace_column_nulls(bf_df_cleaned, 'Recipe Name')

Replaced 11 nulls in Recipe Name
Example replacement: _ No Name 1


In [ ]:
bf_df_cleaned = replace_column_nulls(bf_df_cleaned, 'Recipe Name')
bf_df_cleaned

,Recipe Name,Style,Style Number,OG,Plato,FG,ABV,IBU
0,Avg. Perfect Northeast IPA (NEIPA),Specialty IPA: New England IPA,None,1.062,15.2,1.013,6.50,59.26
1,Sierra Nevada Pale Ale Clone,American Pale Ale,None,1.055,13.6,1.013,5.58,39.79
2,Vanilla Cream Ale,Cream Ale,None,1.055,13.6,1.013,5.48,19.44
3,Zombie Dust Clone - ALL GRAIN,American IPA,None,1.061,15.0,1.016,5.94,62.42
4,Russian River Pliny The Elder (original),Imperial IPA,None,1.072,17.5,1.018,7.09,232.89
...,...,...,...,...,...,...,...,...
215575,Awesome Recipe,American IPA,None,1.054,13.3,1.010,5.82,19.23
215576,Dildo NEIPA,Specialty IPA: New England IPA,None,1.061,15.0,1.012,6.43,18.70
215577,Double Down,Specialty IPA: New England IPA,None,1.082,19.8,1.015,8.83,28.76
215578,AGAIN,American IPA,None,1.029,7.3,1.010,2.51,60.87
